# Day 037 Solution — Data Cleaning

Full cleaning pipeline: type coercion → string normalisation → null handling → deduplication. All data defined inline for headless execution.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import pandas as pd

def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    result   = df.copy()
    if strategy == 'drop':
        return result.dropna().reset_index(drop=True)
    num_cols = result.select_dtypes(include='number').columns
    if strategy == 'zero':
        result[num_cols] = result[num_cols].fillna(0)
    elif strategy == 'mean':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].mean())
    elif strategy == 'median':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].median())
    else:
        raise ValueError(
            f"Unknown strategy {strategy!r}. "
            "Use 'drop', 'zero', 'mean', or 'median'."
        )
    return result


import pandas as pd

def coerce_numeric_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce')
    return result


import pandas as pd

def clean_string_column(df: pd.DataFrame, col: str) -> pd.DataFrame:
    result = df.copy()
    result[col] = result[col].str.strip().str.lower()
    return result


import pandas as pd

def deduplicate(df: pd.DataFrame, subset: list | None = None) -> pd.DataFrame:
    return df.drop_duplicates(subset=subset).reset_index(drop=True)


import pandas as pd

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    for col in result.select_dtypes(include='object').columns:
        result[col] = result[col].str.strip()
    for col in result.select_dtypes(include='number').columns:
        result[col] = result[col].fillna(result[col].median())
    return result.drop_duplicates().reset_index(drop=True)

## Step 1 — Load the Messy Dataset

In [ ]:
MESSY_CSV = (
    'order_id,product,price,quantity,region\n'
    '1, Widget ,25.0,10, North\n'
    '2, Gadget ,bad_price,,South\n'
    '3,Widget,25.0,20,North\n'
    '4, Widget ,25.0,20, North\n'
    '5,Doohickey,8.0,50,EAST\n'
    '5,Doohickey,8.0,50,EAST\n'
    '6,Thingamajig,200.0,8,West'
)
raw = pd.read_csv(io.StringIO(MESSY_CSV))
print(f'Raw shape: {raw.shape}')
print(f'Null counts:\n{raw.isnull().sum()}')
print(f'\nraw dtypes:\n{raw.dtypes}')

assert raw.shape == (7, 5)
assert raw.isnull().sum().sum() > 0

## Step 2 — Coerce Numeric Columns

In [ ]:
df = coerce_numeric_columns(raw, ['price'])
print(f'price dtype after coerce: {df["price"].dtype}')
print(f'Nulls in price: {df["price"].isnull().sum()}')

assert pd.api.types.is_numeric_dtype(df['price'])
assert df['price'].isnull().sum() == 1

## Step 3 — Clean String Columns

In [ ]:
df = clean_string_column(df, 'product')
df = clean_string_column(df, 'region')
print('Products:', df['product'].unique().tolist())
print('Regions :', df['region'].unique().tolist())

assert 'widget' in df['product'].values
assert not any(v.startswith(' ') for v in df['product'].dropna())
assert 'north' in df['region'].values

## Step 4 — Fill Null Values

In [ ]:
df = drop_or_fill_nulls(df, strategy='median')
null_count = df.isnull().sum().sum()
print(f'Nulls after fill: {null_count}')
print(df[['product', 'price', 'quantity', 'region']].to_string())

assert null_count == 0

## Step 5 — Deduplicate

In [ ]:
cleaned = deduplicate(df)
print(f'\nShape after dedup: {cleaned.shape}')
print(cleaned.to_string())

assert len(cleaned) < len(raw)
assert not cleaned.duplicated().any()
assert list(cleaned.index) == list(range(len(cleaned)))

## Step 6 — One-Call Pipeline (clean_dataframe)

In [ ]:
# Same dataset, same result in one call
raw2 = pd.read_csv(io.StringIO(MESSY_CSV))
raw2 = coerce_numeric_columns(raw2, ['price'])
auto_cleaned = clean_dataframe(raw2)
print(f'clean_dataframe result shape: {auto_cleaned.shape}')
print(f'Nulls: {auto_cleaned.isnull().sum().sum()}')
print(f'Dupes: {auto_cleaned.duplicated().sum()}')

assert auto_cleaned.isnull().sum().sum() == 0
assert not auto_cleaned.duplicated().any()

print('\nData Cleaning complete!')